# PyTorch RNN Bridge: From Keras Sequences to Transformer-Ready Tensors

## Keep the melody task. Change the framework. Prepare the next architecture.

Prerequisite 05 established why sequences need memory, how a vanilla RNN is unrolled, why BPTT can lose long-range gradients, and how LSTM gates provide a better route for information. Prerequisite 06 then made token IDs, embeddings, padding, and masked sequence loss explicit. This notebook does **not** repeat that theory.

Instead, it rebuilds the familiar next-token music task in PyTorch, using the tensor, module, and optimizer contracts from prerequisite 03. Framework details become visible: integer token tensors, embeddings, recurrent outputs and states, loss dimensions, gradient clipping, `eval()` mode, and state-carrying generation.

| Part | Keras concept already known | PyTorch bridge | Evidence produced |
|---|---|---|---|
| 1 | Character/music tokens and embeddings | `torch.long` IDs and `nn.Embedding` | `(B, T) -> (B, T, D)` assertions |
| 2 | `SimpleRNN`, `LSTM`, and gate equations | Recurrent return-state API plus fused gate parameters | output, `h_n`, `c_n`, and gate-chunk contracts |
| 3 | Sequence language model | Readable `nn.Module` | `(B, T, V)` logits |
| 4 | `GradientTape`, clipping, and sampling | Explicit loop, `clip_grad_norm_`, temperature, and returned state | finite gradients, falling loss, generated motif |
| 5 | RNN limitations | Transformer contract handoff | exactly what attention retains and changes |

The corpus below is generated symbolic notation, not downloaded music or copied score text. It is deliberately small enough for CPU execution and is original to this notebook.

![End-to-end PyTorch sequence pipeline from symbolic music tokens and long token IDs through embeddings, LSTM outputs and final states, vocabulary logits, padding-aware cross-entropy, and state-carrying autoregressive generation.](images/11.png)

*The architecture changes across the track, but the token, tensor, logits, loss, and generation contracts remain reusable.*

---

## Prerequisite Bridge — Do Not Relearn What You Already Know

| Established in the Keras prerequisite | This notebook translates it into PyTorch | Deferred to `01-transformers` |
|---|---|---|
| `h_t` holds a running summary, BPTT traverses time, LSTM has gates | `nn.LSTM` API, returned `(h_n, c_n)`, parameter layout, loss shape, optimizer loop | Positional encoding, Q/K/V, attention, causal masks, multi-head blocks |
| `(batch, time, features)` sequence contract | `batch_first=True`, `(B,T)` token IDs, `(B,T,D)` embeddings, `(B,T,V)` logits | `(B,H,T,d_h)` head reshaping |
| Teacher forcing and next-token objective | shifted targets, `CrossEntropyLoss`, `ignore_index`, gradient clipping | Transformer decoder training and autoregressive context windows |

### Keras reference — names you already know

```python
layers.Embedding(vocab_size, d_embed)
layers.LSTM(hidden_size, return_sequences=True, return_state=True)
loss_fn = keras.losses.SparseCategoricalCrossentropy(from_logits=True)
with tf.GradientTape() as tape:
    logits = model(token_ids, training=True)
    loss = loss_fn(target_ids, logits)
```

The rest of this notebook keeps the data and objective familiar while translating the execution model.


In [ ]:
#  Setup — PyTorch-only runtime for the bridge
import random
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn

SEED = 23

# Seed Python, NumPy, and PyTorch separately since each owns its own RNG
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# Prefer a GPU if visible to PyTorch, otherwise fall back to CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__} on {device}")

---

## Part 1 — The Same Music Tokens, Now With PyTorch Dtypes

We retain one music-language idea from the prerequisite: a key token establishes a motif, bars organize its phrase, and the next token is the prediction target. The corpus varies phrase length so padding has a concrete job.

### Predict first

What does `nn.Embedding` need as input?

1. `float32` pitch values, because embeddings are vectors.
2. `long` integer IDs, because an embedding is a learnable lookup table.
3. one-hot vectors, because the table needs a vocabulary-width input.


In [ ]:
#  Original symbolic-music corpus — generated locally, no external data required
PAD, BOS, EOS, BAR = "<PAD>", "<BOS>", "<EOS>", "<BAR>"
KEY_TO_TRIAD = {
    "KEY_C": ["C", "E", "G"],
    "KEY_G": ["G", "B", "D"],
    "KEY_F": ["F", "A", "C"],
}
VARIATIONS = [[0, 1, 2, 1, 0], [0, 2, 1, 2, 0], [0, 1, 0, 1, 2, 1, 0], [2, 1, 0, 1, 2]]


# Build one motif: a key token, a triad phrase, a bar marker, then the phrase reversed
def make_motif(key_token, variation):
    triad = KEY_TO_TRIAD[key_token]

    # Map each variation index onto its triad note
    first_bar = [triad[index] for index in variation]
    return [BOS, key_token, *first_bar, BAR, *reversed(first_bar), EOS]


sequences = []

# Generate every key/variation combination, repeated so each pattern has enough examples
for key_token in KEY_TO_TRIAD:
    for variation in VARIATIONS:
        sequences.extend([make_motif(key_token, variation)] * 12)
random.shuffle(sequences)

# Collect every distinct token across all sequences into a sorted vocabulary, plus PAD
vocabulary = sorted({PAD} | {token for sequence in sequences for token in sequence})

# Build the token<->ID lookup tables that anchor every tensor conversion below
token_to_id = {token: index for index, token in enumerate(vocabulary)}
id_to_token = {index: token for token, index in token_to_id.items()}
PAD_ID = token_to_id[PAD]
print(f"Generated sequences: {len(sequences)}")
print(f"Vocabulary ({len(vocabulary)} tokens): {vocabulary}")
print("Example sequence:", " ".join(sequences[0]))
print("-> The familiar next-token task survives; only the framework contract changes.")


In [ ]:
#  Token IDs, shifted targets, and padding — the PyTorch sequence tensor contract
# Convert every token in every sequence to its vocabulary ID
encoded = [[token_to_id[token] for token in sequence] for sequence in sequences]

# Inputs are all-but-the-last token; targets are the same sequence shifted one position later
inputs_raw, targets_raw = [sequence[:-1] for sequence in encoded], [sequence[1:] for sequence in encoded]
max_time = max(len(sequence) for sequence in inputs_raw)


# Right-pad every row to max_time so the batch forms a rectangular tensor
def pad_right(rows, pad_value):
    return torch.tensor([row + [pad_value] * (max_time - len(row)) for row in rows], dtype=torch.long)


token_ids = pad_right(inputs_raw, PAD_ID).to(device)
targets = pad_right(targets_raw, PAD_ID).to(device)

# Track each row's real (unpadded) length for later packing
lengths = torch.tensor([len(row) for row in inputs_raw], dtype=torch.long, device=device)

# Confirm the dtype and shape contract PyTorch's recurrent/loss APIs expect
assert token_ids.dtype == torch.long and targets.dtype == torch.long
assert token_ids.shape == targets.shape
print(f"token_ids: {tuple(token_ids.shape)}, dtype={token_ids.dtype}")
print(f"targets:   {tuple(targets.shape)}, dtype={targets.dtype}")
print(f"lengths:   min={lengths.min().item()}, max={lengths.max().item()}")
print("-> Integer IDs are inputs to a lookup table; PAD is bookkeeping, never a target to learn.")

### Keras reference — embedding lookup

```python
embedded = layers.Embedding(vocab_size, d_embed, mask_zero=True)(token_ids)
# (B, T) integer IDs -> (B, T, D) float vectors
```

In PyTorch, `padding_idx=PAD_ID` keeps the padding row from receiving gradient updates. That is only **loss/embedding masking**: it does not stop an LSTM from stepping over trailing pads. The next section uses `pack_padded_sequence` so final recurrent states genuinely stop at each melody's last real token.


In [ ]:
#  Embedding lookup — prove that every token selects exactly one table row
D_EMBED = 12

# padding_idx keeps the PAD row's embedding from receiving gradient updates
embedding = nn.Embedding(len(vocabulary), D_EMBED, padding_idx=PAD_ID).to(device)
embedded = embedding(token_ids)
batch_index, time_index = 0, 2
lookup_id = token_ids[batch_index, time_index]

# Confirm the embedded vector at one position exactly matches its table row
assert torch.equal(embedded[batch_index, time_index], embedding.weight[lookup_id])
assert embedded.shape == (*token_ids.shape, D_EMBED)
print(f"IDs:        {tuple(token_ids.shape)} = (batch, time)")
print(f"Embeddings: {tuple(embedded.shape)} = (batch, time, embedding_dim)")
print(f"Lookup parity at [0, 2]: {torch.equal(embedded[batch_index, time_index], embedding.weight[lookup_id])}")
print("-> nn.Embedding is a learned table lookup, exactly like the Keras concept you already used.")

#### What just happened — and what's missing

The task has the familiar token-to-vector shape, but the vectors still need an ordered model. The prerequisite already explained why order matters. Here we inspect what PyTorch returns when a recurrent module processes that order.


---

## Part 2 — PyTorch's Recurrent API: Outputs Are Not the Same as Final State

### Keras reference — `return_sequences` and `return_state`

```python
sequence_output, final_h, final_c = layers.LSTM(
    hidden_size, return_sequences=True, return_state=True
)(embedded)
```

PyTorch returns the same ideas as a tuple. The naming and shape layout are what matter now.


In [ ]:
#  RNN and LSTM shape inspection — pack padded rows so final state means final real token
HIDDEN_SIZE = 18
plain_rnn = nn.RNN(D_EMBED, HIDDEN_SIZE, batch_first=True).to(device)
lstm = nn.LSTM(D_EMBED, HIDDEN_SIZE, batch_first=True).to(device)

# Pack so the recurrent step skips trailing PAD positions instead of processing them
packed_embeddings = nn.utils.rnn.pack_padded_sequence(
    embedded, lengths.cpu(), batch_first=True, enforce_sorted=False
)
packed_rnn_output, rnn_h_n = plain_rnn(packed_embeddings)
packed_lstm_output, (lstm_h_n, lstm_c_n) = lstm(packed_embeddings)

# Unpack back to a rectangular (B, T, H) tensor, zero-filling the padded positions
rnn_output, _ = nn.utils.rnn.pad_packed_sequence(packed_rnn_output, batch_first=True, total_length=max_time)
lstm_output, _ = nn.utils.rnn.pad_packed_sequence(packed_lstm_output, batch_first=True, total_length=max_time)

# Confirm per-timestep outputs and final states match the expected (B, T, H) / (layers, B, H) shapes
assert rnn_output.shape == (token_ids.shape[0], token_ids.shape[1], HIDDEN_SIZE)
assert rnn_h_n.shape == (1, token_ids.shape[0], HIDDEN_SIZE)
assert lstm_output.shape == rnn_output.shape
assert lstm_h_n.shape == lstm_c_n.shape == rnn_h_n.shape
print(f"RNN output:       {tuple(rnn_output.shape)} = (B, T, H)")
print(f"RNN final h_n:    {tuple(rnn_h_n.shape)} = (layers, B, H)")
print(f"LSTM output:      {tuple(lstm_output.shape)} = (B, T, H)")
print(f"LSTM final h_n:   {tuple(lstm_h_n.shape)} = (layers, B, H)")
print(f"LSTM final c_n:   {tuple(lstm_c_n.shape)} = (layers, B, H)")
print("-> Packed input stops recurrent state at each row's last real token; h_n/c_n are valid final states.")

![A packed one-layer unidirectional LSTM produces one output per valid timestep; unpacking adds zero-filled positions for shorter sequences, while global final hidden and cell states retain layers-by-batch-by-hidden shape.](images/5.png)

*Packed execution stops at each sequence's final real token; unpacking restores rectangular output shape with zeros rather than additional recurrent computation.*

### Predict first — why do four LSTM gate chunks appear in one PyTorch tensor?

1. PyTorch stores four unrelated layers in one tensor by accident.
2. PyTorch fuses input, forget, candidate, and output gate projections for efficient execution.
3. Each chunk corresponds to a separate music token.


In [ ]:
#  Inspect fused LSTM parameters — map known gates without re-deriving their equations
# List every learnable parameter's name and shape
for parameter_name, parameter in lstm.named_parameters():
    print(f"{parameter_name:18s} {tuple(parameter.shape)}")
gate_names = ["input", "forget", "candidate", "output"]

# Split each fused weight tensor into its four gate-sized chunks, in PyTorch's (i, f, g, o) order
for gate_name, input_chunk, hidden_chunk in zip(
    gate_names, lstm.weight_ih_l0.chunk(4, dim=0), lstm.weight_hh_l0.chunk(4, dim=0)
):
    print(f"{gate_name:9s}: W_ih {tuple(input_chunk.shape)}, W_hh {tuple(hidden_chunk.shape)}")
print("-> PyTorch gate order is (input, forget, candidate, output): i, f, g, o.")
print("-> The prerequisite derived the gates; this cell maps that knowledge to fused PyTorch storage.")

![Two fused PyTorch LSTM weight tensors divided into four aligned chunks in input, forget, candidate, and output gate order.](images/7.png)

*PyTorch stores the four gate projections together in `(i, f, g, o)` order for efficient execution.*

#### What just happened — and what's missing

PyTorch gives us every timestep's output plus the final recurrent state. Packing fixes recurrent-state masking; `ignore_index` will separately mask padded labels in the loss. The fused gate tensor is an implementation detail, not new LSTM theory. We now need to connect every output position to a vocabulary-sized prediction and teach it with the loss contract that the Transformer notebook will reuse.


---

## Part 3 — A Readable PyTorch Music Language Model

### Keras reference — the same model shape

```python
model = keras.Sequential([
    layers.Embedding(vocab_size, d_embed, mask_zero=True),
    layers.LSTM(hidden_size, return_sequences=True),
    layers.Dense(vocab_size),
])
# logits: (B, T, V)
```

The PyTorch class below exposes the same three stages. It is intentionally small enough to read in one pass.


In [ ]:
#  TinyMotifLM — embedding -> LSTM -> vocabulary logits
# Three-stage sequence model: token embedding, LSTM recurrence, then a vocabulary-sized linear head
class TinyMotifLM(nn.Module):
    def __init__(self, vocab_size, d_embed=D_EMBED, hidden_size=HIDDEN_SIZE, pad_id=PAD_ID):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_embed, padding_idx=pad_id)
        self.recurrent = nn.LSTM(d_embed, hidden_size, batch_first=True)
        self.vocabulary_head = nn.Linear(hidden_size, vocab_size)

    def forward(self, ids, lengths=None, state=None):
        vectors = self.embedding(ids)

        # Pack only when lengths are known (training on padded batches); generation calls without them
        if lengths is not None:
            vectors = nn.utils.rnn.pack_padded_sequence(
                vectors, lengths.cpu(), batch_first=True, enforce_sorted=False
            )
            hidden_sequence, next_state = self.recurrent(vectors, state)
            hidden_sequence, _ = nn.utils.rnn.pad_packed_sequence(
                hidden_sequence, batch_first=True, total_length=ids.shape[1]
            )
        else:
            hidden_sequence, next_state = self.recurrent(vectors, state)
        logits = self.vocabulary_head(hidden_sequence)
        return logits, next_state


torch.manual_seed(SEED)
motif_model = TinyMotifLM(len(vocabulary)).to(device)
example_logits, example_state = motif_model(token_ids[:3], lengths=lengths[:3])

# Confirm one vocabulary-sized logit vector per timestep, and a valid final LSTM state
assert example_logits.shape == (3, max_time, len(vocabulary))
assert example_state[0].shape == (1, 3, HIDDEN_SIZE)
print(f"token IDs:    {tuple(token_ids[:3].shape)} = (B, T)")
print(f"model logits: {tuple(example_logits.shape)} = (B, T, V)")
print(f"returned h/c: {tuple(example_state[0].shape)} / {tuple(example_state[1].shape)}")
print("-> Every input timestep receives a vocabulary-sized next-token score vector.")

### Keras reference — sequence cross-entropy

```python
loss_fn = keras.losses.SparseCategoricalCrossentropy(from_logits=True)
loss = loss_fn(target_ids, logits)  # target: (B, T), logits: (B, T, V)
```

PyTorch's `CrossEntropyLoss` expects the class axis in position 1. For sequence logits `(B,T,V)`, transpose to `(B,V,T)`. `ignore_index=PAD_ID` makes padded labels invisible to the objective.


![Tensor-shape pipeline from token IDs through embeddings, LSTM states, and vocabulary logits, followed by the axis permutation from B-T-V to B-V-T required for sequence cross-entropy; padded targets are ignored.](images/8.png)

*The transpose only reorders axes so vocabulary classes occupy position 1; it does not learn or alter values.*

In [ ]:
#  Loss shape contract — logits stay raw; padding is ignored, not predicted
sequence_loss = nn.CrossEntropyLoss(ignore_index=PAD_ID)
raw_logits, _ = motif_model(token_ids, lengths=lengths)

# Transpose logits to (B, V, T) so the class axis sits where CrossEntropyLoss expects it
loss = sequence_loss(raw_logits.transpose(1, 2), targets)
assert raw_logits.shape == (token_ids.shape[0], token_ids.shape[1], len(vocabulary))
assert torch.isfinite(loss)
print(f"raw logits: {tuple(raw_logits.shape)} -> transpose: {tuple(raw_logits.transpose(1, 2).shape)}")
print(f"targets:    {tuple(targets.shape)}, dtype={targets.dtype}, PAD_ID={PAD_ID}")
print(f"loss:       {loss.item():.4f}")
print("-> CrossEntropyLoss consumes raw (B, V, T) logits and long (B, T) targets.")


In [ ]:
#  Causal-invariance proof — future token changes cannot alter earlier unidirectional logits
prefix = [BOS, "KEY_C", "C", "E", BAR]
sequence_a = prefix + ["G", "E", "C", EOS]
sequence_b = prefix + ["C", "G", "E", EOS]
ids_a = torch.tensor([[token_to_id[token] for token in sequence_a]], dtype=torch.long, device=device)
ids_b = torch.tensor([[token_to_id[token] for token in sequence_b]], dtype=torch.long, device=device)
motif_model.eval()

# Run both sequences through the model without building an autograd graph
with torch.no_grad():
    logits_a, _ = motif_model(ids_a)
    logits_b, _ = motif_model(ids_b)
prefix_length = len(prefix)

# Compare only the shared-prefix timesteps between the two diverging sequences
same_prefix_logits = torch.allclose(logits_a[:, :prefix_length], logits_b[:, :prefix_length], atol=1e-6)
assert same_prefix_logits
print(f"Logits through timestep {prefix_length - 1} identical after changing future tokens: {same_prefix_logits}")
print("-> A unidirectional RNN is causal by construction: information flows left to right only.")


#### What just happened — and what's missing

The next-token objective has reached its PyTorch form: `long` token IDs become `(B,T,D)` vectors, an LSTM produces `(B,T,H)`, a linear head produces `(B,T,V)` logits, and cross-entropy scores the shifted targets. Now we train it. The new framework detail is that clipping occurs after `.backward()` exposes real gradients and before `.step()` changes weights.


---

## Part 4 — Explicit Sequence Training and Gradient Clipping

### Keras reference — one custom training step

```python
with tf.GradientTape() as tape:
    logits = model(inputs, training=True)
    loss = loss_fn(targets, logits)
grads = tape.gradient(loss, model.trainable_variables)
grads, _ = tf.clip_by_global_norm(grads, 1.0)
optimizer.apply_gradients(zip(grads, model.trainable_variables))
```

The PyTorch equivalent follows the same logical order, but `.grad` lives on each parameter after `backward()` until the optimizer is told to clear it.


![Six-stage sequence-training iteration showing gradient clipping after backward creates gradients and before the optimizer updates parameters.](images/9.png)

*Measure and clip the global gradient norm after `.backward()` and before `optimizer.step()`.*

In [ ]:
#  Full training loop — four visible operations plus sequence-gradient clipping
torch.manual_seed(SEED)
motif_model = TinyMotifLM(len(vocabulary)).to(device)
optimizer = torch.optim.Adam(motif_model.parameters(), lr=0.03)
MAX_GRAD_NORM = 1.0
EPOCHS = 220
loss_history = []
motif_model.train()

# zero_grad -> forward -> loss -> backward -> clip -> step, once per epoch
for epoch in range(EPOCHS):
    optimizer.zero_grad()
    logits, _ = motif_model(token_ids, lengths=lengths)
    loss = sequence_loss(logits.transpose(1, 2), targets)
    loss.backward()
    unclipped_norm = nn.utils.clip_grad_norm_(motif_model.parameters(), MAX_GRAD_NORM)
    optimizer.step()
    loss_history.append(loss.item())
    if epoch in {0, 19, 99, EPOCHS - 1}:
        print(f"epoch={epoch + 1:3d} loss={loss.item():.4f} gradient_norm_before_clip={float(unclipped_norm):.4f}")

# Plot the training loss curve across every epoch
plt.figure(figsize=(9, 3.5))
plt.plot(loss_history, color="#4C78A8")
plt.title("TinyMotifLM: real training loss on the generated corpus")
plt.xlabel("Epoch")
plt.ylabel("Cross-entropy")
plt.tight_layout()
plt.show()

# Confirm gradient clipping kept every parameter finite through all 220 epochs
assert all(torch.isfinite(parameter).all() for parameter in motif_model.parameters())
print(f"Loss: {loss_history[0]:.3f} -> {loss_history[-1]:.3f}")
print("-> The PyTorch training loop is the Keras custom loop made inspectable, including where clipping belongs.")


### Your turn — temperature is an inference knob, not a training rewrite

Change `TEMPERATURE` below, predict whether the next-token choices become more varied, and compare the generated continuations. The trained weights stay fixed.


In [ ]:
#  State-carrying autoregressive generation — eval mode, no graph, one new token at a time
TEMPERATURE = 0.7  # CHANGE: try 0.3 or 1.2


# Generate new tokens one at a time, carrying LSTM state forward instead of replaying the prefix
def generate(seed_tokens, max_new_tokens=16, temperature=TEMPERATURE):
    motif_model.eval()
    generated = list(seed_tokens)
    state = None

    # Disable autograd tracking since generation is inference-only
    with torch.no_grad():
        seed_ids = torch.tensor([[token_to_id[token] for token in seed_tokens]], dtype=torch.long, device=device)
        logits, state = motif_model(seed_ids, state=state)
        next_logits = logits[:, -1, :]

        # Sample one new token at a time, feeding each prediction back in as the next input
        for _ in range(max_new_tokens):
            probabilities = torch.softmax(next_logits / temperature, dim=-1)
            next_id = torch.multinomial(probabilities, num_samples=1)
            next_token = id_to_token[next_id.item()]
            generated.append(next_token)

            # Stop early once the model emits the end-of-sequence token
            if next_token == EOS:
                break
            logits, state = motif_model(next_id, state=state)
            next_logits = logits[:, -1, :]
    return generated


# Generate one continuation per key to sample a few distinct motifs
for key in ("KEY_C", "KEY_G", "KEY_F"):
    continuation = generate([BOS, key])
    print(f"{key}: {' '.join(continuation)}")
print("-> Returned LSTM state lets generation continue one token at a time without replaying the whole prefix.")


#### Watch generation carry state instead of replaying history

![Animation of an LSTM generating tokens one at a time while carrying its hidden and cell states into each next step](images/lstm-state-carrying-generation.gif)

The seed is processed once. After that, each sampled token becomes the next input while `(h, c)` moves forward as the compressed history. Unlike a Transformer with a growing context or KV cache, this loop does not revisit every earlier token.

#### What just happened — and what the next architecture fixes

The PyTorch model trained and generated by carrying the returned `(h, c)` state forward. This is the exact Keras sequence task you already understood, now with explicit PyTorch mechanics.

But each new output depends on a compressed state passed through every prior timestep. The prerequisite already measured why long recurrent paths strain gradients; this notebook has shown the corresponding PyTorch interface. The next chapter keeps token IDs, embeddings, `(B,T,V)` logits, cross-entropy, and autoregressive generation — but replaces recurrence with direct token-to-token attention.


---

## Part 5 — Transformer Handoff: Keep the Contract, Replace the Path

| This PyTorch RNN bridge | `01-transformers` changes | What stays true |
|---|---|---|
| `nn.Embedding`: `(B,T) -> (B,T,D)` | Adds positional information and attention projections | Token IDs are `long`; embeddings are float vectors |
| LSTM recurrent path and `(h,c)` state | Each token can directly weigh earlier tokens with a causal mask | Sequence outputs still carry a time axis |
| `nn.Linear(H,V)` produces logits `(B,T,V)` | Transformer block produces logits `(B,T,V)` | Raw logits feed `CrossEntropyLoss` |
| `zero_grad -> forward -> backward -> step` | Same optimization pattern through attention layers | Autograd and optimizer mechanics are unchanged |
| One-token-at-a-time generation | One-token-at-a-time generation with a growing context/KV cache | `eval()`, `no_grad()`, softmax, sampling remain inference tools |

No Q/K/V, positional encoding, or causal-mask matrix is implemented here. Those mechanisms are the subject of `01-transformers`, where the pressure created by recurrence becomes their motivation.


![Comparison of sequential information flow through an RNN or LSTM state with causal direct connections in a Transformer, above a shared rail of token, embedding, logits, loss, optimization, and autoregressive-generation contracts.](images/10.png)

*Transformers replace the recurrent information path while preserving the surrounding language-modeling contracts.*

---

## Summary — The PyTorch Sequence Contract You Now Own

| Step | Evidence you produced | PyTorch habit to retain |
|---|---|---|
| Tokenize and pad | `token_ids` / `targets` are `long`, shape `(B,T)` | Tokens are IDs; padding is bookkeeping |
| Embed | `(B,T) -> (B,T,D)` lookup parity assertion | `nn.Embedding` is a learned table |
| Recur | `output: (B,T,H)`, final `h_n` / `c_n` | `batch_first=True` keeps shape reasoning readable |
| Predict | logits `(B,T,V)` | Keep logits raw until loss/inference |
| Learn | finite gradients and falling loss | `zero_grad -> backward -> clip -> step` |
| Generate | state-carrying token loop | `eval()` + `no_grad()` isolate inference |
| Hand off | RNN path vs attention path table | The sequence/loss contract survives the architecture change |

### Key insights to keep

- **PyTorch's recurrent API separates every timestep's output from final state:** `output` is `(B,T,H)`; `h_n` and `c_n` carry the ending state.
- **The Keras next-token task translates cleanly:** only tensor layouts, dtypes, and explicit optimizer steps change.
- **`CrossEntropyLoss` wants raw logits with the class dimension in position 1:** `(B,T,V)` becomes `(B,V,T)`.
- **`ignore_index` says padding is not music:** a padded target must contribute no loss.
- **The RNN's pressure becomes the Transformer's motivation:** token and loss contracts remain; the information path changes.

**Next:** [`../../genai/01-transformers/01-attention-and-transformer-blocks.ipynb`](../../genai/01-transformers/01-attention-and-transformer-blocks.ipynb) builds direct attention from the tensor, autograd, loss, and generation habits you just exercised.